# UMTAM Curvature-Aware Pruning: CoLA (Linguistic Acceptability)

**Task:** Binary classification - Is the sentence grammatically acceptable?

**Dataset:** CoLA (Corpus of Linguistic Acceptability)
- Train: 8,551 sentences
- Validation: 1,043 sentences
- Metric: Matthews Correlation Coefficient (MCC)

**Model:** BERT-base-uncased (110M parameters)

**Hypothesis:** Curvature-aware pruning outperforms magnitude-only pruning, especially at low densities (1-10%)

**Expected Runtime:** ~45-60 minutes

---
## 📦 Setup

In [ ]:
# Install packages
!pip install -q transformers datasets torch evaluate matplotlib seaborn scikit-learn
print("✓ Packages installed")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import evaluate
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"✓ Device: {device}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## ⚙️ Configuration

In [ ]:
CONFIG = {
    # Model
    'model_name': 'bert-base-uncased',
    'num_labels': 2,
    
    # Training
    'max_length': 128,
    'batch_size': 32,
    'num_epochs': 4,
    'lr': 2e-5,
    
    # Pruning
    'densities': [1, 5, 10, 20, 40, 60, 90],
    
    # Metric
    'metric': 'matthews_correlation',  # MCC is standard for CoLA
    
    # Other
    'seed': 42,
}

print("✓ Configuration:")
print(f"  Model: {CONFIG['model_name']}")
print(f"  Metric: {CONFIG['metric']}")
print(f"  Densities: {CONFIG['densities']}")
print(f"  Epochs: {CONFIG['num_epochs']}")

---
## 📚 Load Dataset

In [ ]:
print("Loading CoLA dataset...")

# Load tokenizer and dataset
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
dataset = load_dataset('glue', 'cola')

print(f"\nDataset sizes:")
print(f"  Train: {len(dataset['train'])} sentences")
print(f"  Validation: {len(dataset['validation'])} sentences")

# Tokenize
def tokenize_function(examples):
    result = tokenizer(
        examples['sentence'],
        padding='max_length',
        truncation=True,
        max_length=CONFIG['max_length']
    )
    result['labels'] = examples['label']
    return result

tokenized = dataset.map(tokenize_function, batched=True)

train_dataset = tokenized['train'].shuffle(seed=CONFIG['seed'])
eval_dataset = tokenized['validation']

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
eval_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

print(f"\n✓ Dataset prepared")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Eval: {len(eval_dataset)} samples")

---
## 🏋️ Training Function

In [ ]:
def train_and_collect_curvature(model, train_dataset):
    """
    Train model with AdamW and collect curvature statistics.
    Returns: trained model, initial weights, curvature-aware saliency scores
    """
    model.to(device)
    model.train()
    
    optimizer = AdamW(model.parameters(), lr=CONFIG['lr'])
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
    
    # Store initial weights (w0)
    w0_dict = {}
    for name, param in model.named_parameters():
        if len(param.shape) == 2:  # Only weight matrices
            w0_dict[name] = param.data.clone().cpu()
    
    print(f"Stored initial weights for {len(w0_dict)} layers")
    
    # Initialize curvature tracking (factorized: R for rows, C for columns)
    curvature_dict = {}
    for name, param in model.named_parameters():
        if len(param.shape) == 2:
            m, n = param.shape
            curvature_dict[name] = {
                'R': torch.zeros(m, device=device),  # Row-wise curvature
                'C': torch.zeros(n, device=device),  # Column-wise curvature
            }
    
    print(f"Tracking curvature for {len(curvature_dict)} layers")
    
    # Training loop
    print(f"\nTraining for {CONFIG['num_epochs']} epochs...")
    for epoch in range(CONFIG['num_epochs']):
        total_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['num_epochs']}")
        
        for batch in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}
            
            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            
            # Accumulate gradient statistics (curvature proxy)
            with torch.no_grad():
                for name, param in model.named_parameters():
                    if name in curvature_dict and param.grad is not None:
                        grad_sq = param.grad ** 2
                        curvature_dict[name]['R'] += grad_sq.sum(dim=1)
                        curvature_dict[name]['C'] += grad_sq.sum(dim=0)
            
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_loss = total_loss / len(train_loader)
        print(f"  Epoch {epoch+1} average loss: {avg_loss:.4f}")
    
    # Compute curvature-aware saliency scores
    print("\nComputing saliency scores...")
    saliency_dict = {}
    
    for name in curvature_dict.keys():
        param = dict(model.named_parameters())[name]
        w0 = w0_dict[name].to(device)
        
        # Parameter change: (w - w0)²
        param_change = (param.data - w0) ** 2
        
        # Curvature approximation: sqrt(R ⊗ C)
        R = curvature_dict[name]['R'].clamp(min=1e-8)
        C = curvature_dict[name]['C'].clamp(min=1e-8)
        curvature_approx = torch.sqrt(R.unsqueeze(1) * C.unsqueeze(0))
        
        # Saliency = parameter_change × curvature
        saliency_dict[name] = (param_change * curvature_approx).cpu()
    
    print(f"✓ Training complete. Computed saliency for {len(saliency_dict)} layers.")
    return model, w0_dict, saliency_dict

---
## 🔪 Pruning Functions

In [ ]:
def apply_pruning_inplace(model, w0_dict, saliency_dict, density_pct, use_curvature=True):
    """
    Apply Fast Fisher Grafting in-place:
    - Keep top-saliency parameters
    - Revert low-saliency parameters to initialization (w0)
    
    If use_curvature=False, uses magnitude-only saliency: (w - w0)²
    """
    # Collect all saliency scores
    all_scores = []
    for name in saliency_dict.keys():
        if use_curvature:
            # Full curvature-aware saliency
            scores = saliency_dict[name].flatten()
        else:
            # Magnitude-only: just parameter change
            param = dict(model.named_parameters())[name]
            w0 = w0_dict[name].to(device)
            scores = ((param.data - w0) ** 2).cpu().flatten()
        
        all_scores.append(scores)
    
    all_scores = torch.cat(all_scores)
    
    # Determine threshold for keeping top density_pct
    k = int(len(all_scores) * (density_pct / 100.0))
    threshold = torch.kthvalue(all_scores, len(all_scores) - k).values
    
    # Apply grafting
    total_params = 0
    kept_params = 0
    
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in saliency_dict:
                w0 = w0_dict[name].to(device)
                
                if use_curvature:
                    scores = saliency_dict[name].to(device)
                else:
                    scores = (param.data - w0) ** 2
                
                # Create mask: 1 if keep, 0 if revert to w0
                mask = (scores > threshold).float()
                
                # Graft: param = mask * param + (1-mask) * w0
                param.data = mask * param.data + (1 - mask) * w0
                
                total_params += mask.numel()
                kept_params += mask.sum().item()
    
    actual_density = 100.0 * kept_params / total_params if total_params > 0 else 0
    print(f"  Pruned to {actual_density:.1f}% density (target: {density_pct}%)")

def restore_weights(model, w_trained_dict):
    """Restore model to trained state (before pruning)"""
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in w_trained_dict:
                param.data.copy_(w_trained_dict[name].to(device))

print("✓ Pruning functions defined")

---
## 📊 Evaluation Function

In [ ]:
def evaluate_model(model, eval_dataset):
    """Evaluate model on validation set"""
    model.eval()
    model.to(device)
    
    eval_loader = DataLoader(eval_dataset, batch_size=CONFIG['batch_size'])
    metric = evaluate.load('glue', 'cola')
    
    with torch.no_grad():
        for batch in eval_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            predictions = torch.argmax(outputs.logits, dim=-1)
            metric.add_batch(predictions=predictions, references=batch['labels'])
    
    result = metric.compute()
    return result[CONFIG['metric']]

print("✓ Evaluation function defined")

---
## 🚀 Run Experiment

In [ ]:
print("="*70)
print("COLA EXPERIMENT: Curvature-Aware Pruning")
print("="*70)

# Load model
print("\nLoading model...")
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels=CONFIG['num_labels']
)
print(f"✓ Loaded {CONFIG['model_name']}")

# Train and collect curvature
model, w0_dict, saliency_dict = train_and_collect_curvature(model, train_dataset)

# Save trained weights (for restoring after pruning)
w_trained_dict = {}
for name, param in model.named_parameters():
    if name in w0_dict:
        w_trained_dict[name] = param.data.clone().cpu()
print("✓ Saved trained weights")

# Evaluate full model
print("\n" + "="*70)
print("Evaluating full model...")
print("="*70)
full_perf = evaluate_model(model, eval_dataset)
print(f"\nFull model {CONFIG['metric'].upper()}: {full_perf:.4f}")

# Test pruning at different densities
results = {
    'umtam': [],
    'magnitude': [],
    'densities': CONFIG['densities'],
    'full_performance': full_perf,
}

print("\n" + "="*70)
print("Testing pruning at different densities...")
print("="*70)

for density in CONFIG['densities']:
    print(f"\n{'─'*70}")
    print(f"Density: {density}%")
    print(f"{'─'*70}")
    
    # Test curvature-aware (UMTAM)
    print("\n[1/2] Curvature-aware (UMTAM)...")
    apply_pruning_inplace(model, w0_dict, saliency_dict, density, use_curvature=True)
    perf_umtam = evaluate_model(model, eval_dataset)
    results['umtam'].append(perf_umtam)
    print(f"  → MCC: {perf_umtam:.4f}")
    
    # Restore to trained state
    restore_weights(model, w_trained_dict)
    
    # Test magnitude-only
    print("\n[2/2] Magnitude-only...")
    apply_pruning_inplace(model, w0_dict, saliency_dict, density, use_curvature=False)
    perf_mag = evaluate_model(model, eval_dataset)
    results['magnitude'].append(perf_mag)
    print(f"  → MCC: {perf_mag:.4f}")
    
    # Compare
    diff = perf_umtam - perf_mag
    if abs(perf_mag) > 0.01:
        improvement = (diff / abs(perf_mag)) * 100
        print(f"\n  📊 Comparison: {diff:+.4f} ({improvement:+.1f}%)")
        if diff > 0.01:
            print(f"  ✓ UMTAM better")
        elif diff < -0.01:
            print(f"  ✗ Magnitude better")
    
    # Restore for next iteration
    restore_weights(model, w_trained_dict)

print("\n" + "="*70)
print("✓ EXPERIMENT COMPLETE!")
print("="*70)

---
## 📊 Results & Visualization

In [ ]:
# Print results table
print("\n" + "="*70)
print("RESULTS SUMMARY: CoLA")
print("="*70)
print(f"\nFull model MCC: {results['full_performance']:.4f}")
print("\nDensity |   UMTAM   | Magnitude | Difference | Improvement")
print("─" * 70)

for i, density in enumerate(results['densities']):
    umtam = results['umtam'][i]
    mag = results['magnitude'][i]
    diff = umtam - mag
    
    if abs(mag) > 0.01:
        pct = (diff / abs(mag)) * 100
    else:
        pct = 0
    
    symbol = "✓" if diff > 0.01 else "✗" if diff < -0.01 else "≈"
    print(f"{density:>3}%    | {umtam:>8.4f}  | {mag:>8.4f}  | {diff:>+9.4f} | {pct:>+7.1f}%  {symbol}")

# Calculate average improvement at low densities
low_density_improvements = []
for i in range(min(3, len(results['densities']))):
    if abs(results['magnitude'][i]) > 0.01:
        imp = (results['umtam'][i] - results['magnitude'][i]) / abs(results['magnitude'][i]) * 100
        low_density_improvements.append(imp)

if low_density_improvements:
    avg_imp = np.mean(low_density_improvements)
    print(f"\nAverage improvement at 1-10% density: {avg_imp:+.2f}%")
    if avg_imp > 20:
        print("✓✓ EXCELLENT - Strong curvature advantage!")
    elif avg_imp > 10:
        print("✓ STRONG - Clear curvature benefit")
    elif avg_imp > 5:
        print("✓ GOOD - Moderate improvement")

In [ ]:
# Plot results
plt.figure(figsize=(10, 6))

plt.plot(results['densities'], results['umtam'], 'o-', 
         label='Curvature-aware (UMTAM)', 
         linewidth=2.5, markersize=10, color='#2E86AB')
plt.plot(results['densities'], results['magnitude'], 's--', 
         label='Magnitude-only', 
         linewidth=2.5, markersize=10, color='#A23B72')
plt.axhline(y=results['full_performance'], color='gray', 
            linestyle=':', label='Full Model', linewidth=2)

# Annotate improvements at low densities
for i in [0, 1, 2]:  # 1%, 5%, 10%
    if i < len(results['densities']) and abs(results['magnitude'][i]) > 0.01:
        improvement = (results['umtam'][i] - results['magnitude'][i]) / abs(results['magnitude'][i]) * 100
        if abs(improvement) > 5:
            sign = '+' if improvement > 0 else ''
            plt.text(results['densities'][i], results['umtam'][i], 
                    f'{sign}{improvement:.0f}%', 
                    fontsize=10, ha='center', va='bottom', fontweight='bold')

plt.xlabel('Density (%)', fontsize=13, fontweight='bold')
plt.ylabel('Matthews Correlation (MCC)', fontsize=13, fontweight='bold')
plt.title('CoLA: Curvature-Aware vs Magnitude Pruning', fontsize=15, fontweight='bold')
plt.legend(fontsize=11, loc='best')
plt.grid(True, alpha=0.3)
plt.xscale('log')
plt.tight_layout()

plt.savefig('cola_pruning_results.png', dpi=300, bbox_inches='tight')
print("\n✓ Saved: cola_pruning_results.png")
plt.show()

# Save results as JSON
with open('cola_pruning_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("✓ Saved: cola_pruning_results.json")

print("\n" + "="*70)
print("All files saved. Experiment complete!")
print("="*70)